In [1]:
#Shishir Khanal
#Sept 11, 2026
#Lunar lander using Soft Actor Critic algorithm using Tensorflow

In [2]:
import gym
import pygame
from collections import UserDict

registry = UserDict(gym.envs.registration.registry)
registry.env_specs = gym.envs.registration.registry
gym.envs.registration.registry = registry

import pybullet_envs
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
import tensorflow_probability as tfp
import tensorflow_probability as tfp
import matplotlib.pyplot as plt

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
2026-09-23 09:55:54.773796: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
#Replay buffer: Fixed size, overwrites early memories with new ones
class ReplayBuffer():
    def __init__(self, max_size, input_shape, n_actions):
        self.mem_size = max_size
        self.mem_cntr = 0
        self.state_memory = np.zeros((self.mem_size, *input_shape)) #'*' means unpack list of elements
        self.new_state_memory = np.zeros((self.mem_size, *input_shape))
        self.action_memory = np.zeros((self.mem_size, n_actions)) #For lunar lander 4 element
        self.reward_memory = np.zeros(self.mem_size)
        self.terminal_memory = np.zeros(self.mem_size, dtype=bool) #mask for setting critique values for new states to 0

    def store_transition(self, state, action, reward, state_, done):
        index = self.mem_cntr % self.mem_size
        self.state_memory[index] = state
        self.action_memory[index] = action
        self.reward_memory[index] = reward
        self.new_state_memory[index] = state_
        self.terminal_memory[index] = done
        self.mem_cntr += 1

    def sample_buffer(self, batch_size):
        max_mem = min(self.mem_cntr, self.mem_size)
        batch = np.random.choice(max_mem, batch_size)
        states = self.state_memory[batch] 
        actions = self.action_memory[batch] 
        rewards = self.reward_memory[batch]
        states_ = self.new_state_memory[batch]
        dones = self.terminal_memory[batch]

        return states, actions, rewards, states_, dones 

In [4]:
class CriticNetwork(keras.Model):
    def __init__(self, fc1_dims=256, fc2_dims=256):
        super(CriticNetwork, self).__init__()
        self.fc1 = Dense(fc1_dims, activation='relu', name='fc1')
        self.fc2 = Dense(fc2_dims, activation='relu', name='fc2')
        self.q = Dense(1, activation=None, name='q')
    
    def call(self, inputs):    #inputs ~ (state, action)
        state, action = inputs
        x = self.fc1(tf.concat([state, action], axis=1))
        x = self.fc2(x)
        q = self.q(x)
        return q

In [5]:
class ValueNetwork(keras.Model):
    def __init__(self, fc1_dims=256, fc2_dims=256):
        super(ValueNetwork, self).__init__()
        self.fc1 = Dense(fc1_dims, activation='relu', name='fc1')
        self.fc2 = Dense(fc2_dims, activation='relu', name='fc2')
        self.v = Dense(1, activation=None, name='v')

    def call(self, state):
        x = self.fc1(state)
        x = self.fc2(x)
        x = self.v(x)
        return x

In [6]:
class ActorNetwork(keras.Model):
    def __init__(self, n_actions, max_action, fc1_dims=256, fc2_dims=256):
        super(ActorNetwork, self).__init__()
        self.fc1 = Dense(fc1_dims, activation='relu', name='fc1')
        self.fc2 = Dense(fc2_dims, activation='relu', name='fc2')
        self.mu = Dense(n_actions, activation='tanh', name='mu')
        self.sigma = Dense(n_actions, activation=None)

        self.max_action = max_action
        self.noise = 1e-6
        
    def call(self, state):
        x = self.fc1(state)
        x = self.fc2(x)
        mu = self.mu(x)
        sigma = self.sigma(x)
        sigma = tf.clip_by_value(sigma, self.noise, 1)
        return mu, sigma

In [7]:
class Agent:
    def __init__(self, alpha=0.0003, beta=0.0003, gamma=0.99, n_actions=4, fc1_dims=256, fc2_dims=256,
                 reward_scale=2, max_size=1_000_000, batch_size=256,
                 tau=0.005, input_dims=[8], env=None, chkpt_dir ='models/'):
        self.gamma = gamma
        self.alpha = alpha
        self.beta = beta
        self.tau = tau
        self.batch_size = batch_size
        self.n_actions = n_actions
        self.max_action = env.action_space.high[0]
        self.min_action = env.action_space.low[0]
        self.chkpt_dir = chkpt_dir
        self.scale = reward_scale
        
        self.actor = ActorNetwork(n_actions=n_actions, fc1_dims=fc1_dims, fc2_dims=fc2_dims,
                                  max_action=self.max_action)
        self.critic_1 = CriticNetwork(fc1_dims=fc1_dims, fc2_dims=fc2_dims)
        self.critic_2 = CriticNetwork(fc1_dims=fc1_dims, fc2_dims=fc2_dims)

        self.value = ValueNetwork(fc1_dims=fc1_dims, fc2_dims=fc2_dims)
        self.target_value = ValueNetwork(fc1_dims=fc1_dims, fc2_dims=fc2_dims)
        
        self.actor.compile(optimizer=Adam(learning_rate=alpha))
        self.critic_1.compile(optimizer=Adam(learning_rate=beta))
        self.critic_2.compile(optimizer=Adam(learning_rate=beta))
        self.value.compile(optimizer=Adam(learning_rate=beta))
        self.target_value.compile(optimizer=Adam(learning_rate=beta))

        self.memory = ReplayBuffer(max_size, input_dims, n_actions)
        self.update_network_parameters(tau=1)

    def update_network_parameters(self, tau=None):
        if tau is None:
            tau = self.tau

        weights = []
        targets = self.target_value.weights
        for i, weight in enumerate(self.value.weights):
            weights.append(weight * tau + targets[i] * (1 - tau))
        self.target_value.set_weights(weights)
        
    def save_models(self):
        if self.memory.mem_cntr > self.batch_size:
            self.actor.save_weights(self.chkpt_dir + 'actor.weights.h5')
            self.critic_1.save_weights(self.chkpt_dir + 'critic_1.weights.h5')
            self.critic_2.save_weights(self.chkpt_dir + 'critic_2.weights.h5')
            self.value.save_weights(self.chkpt_dir + 'value.weights.h5')
            self.target_value.save_weights(self.chkpt_dir + 'target_value.weights.h5')
            print('Model saved successfully...')

    def load_models(self):
        dummy_state = tf.zeros((1, 8), dtype=tf.float32)
        dummy_action = tf.zeros((1, 2), dtype=tf.float32)
    
        self.actor(dummy_state, training=False)
        self.critic_1((dummy_state, dummy_action), training=False)
        self.critic_2((dummy_state, dummy_action), training=False)
        self.value(dummy_state, training=False)
        self.target_value((dummy_state, dummy_action), training=False)
    
        self.actor.load_weights(self.chkpt_dir + 'actor.weights.h5')
        self.critic_1.load_weights(self.chkpt_dir + 'critic_1.weights.h5')
        self.critic_2.load_weights(self.chkpt_dir + 'critic_2.weights.h5')
        self.value.load_weights(self.chkpt_dir + 'value.weights.h5')
        self.target_value.load_weights(self.chkpt_dir + 'target_value.weights.h5')
        print('Models loaded successfully...')

    def store_transition(self, state, action, reward, new_state, done):
        self.memory.store_transition(state, action, reward, new_state, done)

    def sample_normal(self, state):
        mu, sigma = self.actor(state)
        probabilities = tfp.distributions.Normal(mu, sigma)
        actions = probabilities.sample()
        action = tf.math.tanh(actions) * self.actor.max_action
        log_probs = probabilities.log_prob(actions)
        log_probs -= tf.math.log(1 - tf.math.pow(action, 2) + self.actor.noise)
        log_probs = tf.math.reduce_sum(log_probs, axis=1, keepdims=True)
        return action, log_probs
    
    def choose_action(self, observation):
        state = tf.convert_to_tensor([observation])
        actions,_ = self.sample_normal(state)
        
        return actions[0]

    def learn(self):
        if self.memory.mem_cntr < self.batch_size:
            return

        state, action, reward, new_state, done = self.memory.sample_buffer(self.batch_size)
        
        states = tf.convert_to_tensor(state, dtype=tf.float32)
        states_ = tf.convert_to_tensor(new_state, dtype=tf.float32)
        rewards = tf.convert_to_tensor(reward, dtype=tf.float32)
        actions = tf.convert_to_tensor(action, dtype=tf.float32)

        with tf.GradientTape() as tape:   #Critic Loss
            value = tf.squeeze(self.value(states), 1)

            current_policy_actions, log_probs = self.sample_normal(states)
            log_probs = tf.squeeze(log_probs, 1)
            q1_new_pi = self.critic_1((states, current_policy_actions))
            q2_new_pi = self.critic_2((states, current_policy_actions))
            critic_value = tf.squeeze(tf.math.minimum(q1_new_pi, q2_new_pi), 1)
            value_target = critic_value - log_probs
            value_loss = 0.5 * keras.losses.MSE(value, value_target)
            
        params = self.value.trainable_variables
        grads = tape.gradient(value_loss, params)
        
        self.value.optimizer.apply_gradients(zip(grads, params))
        
        with tf.GradientTape() as tape:
            new_policy_actions, log_probs = self.sample_normal(states)
            log_probs = tf.squeeze(log_probs, 1)
            q1_new_policy = self.critic_1((states, new_policy_actions))
            q2_new_policy = self.critic_2((states, new_policy_actions))
            critic_value = tf.squeeze(tf.math.minimum(q1_new_policy, q2_new_policy), 1)
            actor_loss = log_probs - critic_value
            actor_loss = tf.math.reduce_mean(actor_loss)
            
        params = self.actor.trainable_variables
        grads = tape.gradient(actor_loss, params)
        self.actor.optimizer.apply_gradients(zip(grads, params))

        # 2 critics to optimize
        with tf.GradientTape(persistent=True) as tape:
           value_ = tf.squeeze(self.target_value(states_), 1)
           q_hat = self.scale * rewards + self.gamma * value_ * (1-done)
           q1_old_policy = tf.squeeze(self.critic_1((states, actions)), 1)
           q2_old_policy = tf.squeeze(self.critic_2((states, actions)), 1)
           critic_1_loss = 0.5 * keras.losses.MSE(q1_old_policy, q_hat)
           critic_2_loss = 0.5 * keras.losses.MSE(q2_old_policy, q_hat)
            
        params_1 = self.critic_1.trainable_variables
        params_2 = self.critic_2.trainable_variables
        grads_1 = tape.gradient(critic_1_loss, params_1)
        grads_2 = tape.gradient(critic_2_loss, params_2)

        self.critic_1.optimizer.apply_gradients(zip(grads_1, params_1))
        self.critic_2.optimizer.apply_gradients(zip(grads_2, params_2)) 
        
        self.update_network_parameters()

In [8]:
def plot_learning_curve(scores, x, figure_file):
    running_avg = np.zeros(len(scores))
    for i in range(len(running_avg)):
        running_avg[i] = np.mean(scores[max(0, i-100):(i+1)])
    plt.plot(x, running_avg)
    plt.title('Running average of previous 100 scores')
    plt.show()
    plt.savefig(figure_file)

In [ ]:
if __name__ == '__main__':
    #env = gym.make('InvertedPendulumBulletEnv-v0', render_mode="human")
    env = gym.make('InvertedPendulumBulletEnv-v0').unwrapped
    agent = Agent(input_dims=env.observation_space.shape, 
                  n_actions=env.action_space.shape[0], env=env)
    n_games = 250
    load_checkpoint = False
    best_score = env.reward_range[0]
    if load_checkpoint:
        n_games = 1
        agent.load_models()
    filename = 'InvertedPendulum_alpha_' + str(agent.alpha) + '_beta_' + str(agent.beta) +'_' + str(n_games) + '_games_'
    figure_file = 'plots/'+filename + '.png'
    scores = []
    for i in range(n_games):
        done = False
        observation = env.reset()
        score = 0
        while not done:
            action = agent.choose_action(observation)
            observation_, reward, done, info = env.step(action)
            score += reward
            if not load_checkpoint:
                agent.store_transition(observation, action, reward, observation_, done)
                agent.learn()
            observation = observation_ #current state = new state
        scores.append(score)
        
        avg_score = np.mean(scores[-100:])
        #env.render()
        if avg_score > best_score:
            if not load_checkpoint:
                agent.save_models()
            best_score = avg_score

        print('episode {} score {:.1f} average score {:.1f}'.format(i, score, avg_score))

    x = [i+1 for i in range(n_games)]

In [ ]:
plot_learning_curve(scores=scores, x=x, figure_file=figure_file)